# 01. 지도학습 첫걸음 — 회귀와 분류

**대응 강의:** [1강 학습의 종류](../../Lecture/03-머신러닝-기본개념/01-학습의-종류.md), [2강 문제 정의](../../Lecture/03-머신러닝-기본개념/02-문제-정의.md)

이 노트북에서 배우는 것:
- 데이터가 **특성(X)** 과 **타깃(y)** 으로 나뉘는 모습을 눈으로 확인
- scikit-learn의 일관된 **`fit` → `predict`** API 첫 체험
- **회귀**(숫자 예측)와 **분류**(범주 예측)의 차이를 코드로 비교

> ▶️ 상단 메뉴 **런타임 → 모두 실행** 으로 한 번에 돌려보거나, 셀마다 `Shift+Enter`.

In [ ]:
# Colab에는 아래 라이브러리가 이미 설치돼 있습니다. (설치 불필요)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes, load_breast_cancer

print('준비 완료! numpy', np.__version__)

## Part A. 회귀(Regression) — "얼마나?"를 예측

당뇨병 진행도 데이터셋을 씁니다. 여러 특성으로 **1년 뒤 질병 진행도(연속 숫자)** 를 예측 → 타깃이 숫자이므로 **회귀**.

In [ ]:
diabetes = load_diabetes(as_frame=True)
df = diabetes.frame

print('전체 모양 (행=샘플, 열=특성+타깃):', df.shape)
print('특성(feature) 이름:', list(diabetes.feature_names))
print('타깃(target): 질병 진행도 (연속 숫자)')
df.head()

In [ ]:
# 특성 X 와 타깃 y 분리 — 모든 지도학습의 출발점
X = diabetes.data      # 입력: 여러 특성
y = diabetes.target    # 정답: 우리가 맞히려는 값

print('X(특성) 모양:', X.shape, '  <- 442명 x 10개 특성')
print('y(타깃) 모양:', y.shape, '  <- 442명의 정답')
print('타깃 예시 5개:', y[:5].round(1).tolist())

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# 1) 훈련/테스트 분할 (4강에서 '왜 나누는가'를 배웁니다)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2) 모델 생성 → fit(학습) → predict(예측)  ★ scikit-learn 표준 흐름
model = LinearRegression()
model.fit(X_train, y_train)          # 학습: 데이터로 다이얼(파라미터) 조정
pred = model.predict(X_test)         # 예측: 처음 보는 데이터에 적용

# 3) 평가 (회귀 지표 — 로드맵 6번에서 자세히)
print('RMSE (낮을수록 좋음):', mean_squared_error(y_test, pred) ** 0.5)
print('R^2  (1에 가까울수록 좋음):', round(r2_score(y_test, pred), 3))

In [ ]:
# 예측값 vs 실제값 시각화: 점이 대각선에 가까울수록 잘 맞힌 것
plt.figure(figsize=(5, 5))
plt.scatter(y_test, pred, alpha=0.6)
lims = [y_test.min(), y_test.max()]
plt.plot(lims, lims, 'r--', label='완벽한 예측 (y=x)')
plt.xlabel('실제값 (정답)')
plt.ylabel('예측값')
plt.title('회귀: 예측 vs 실제')
plt.legend(); plt.show()

## Part B. 분류(Classification) — "어느 것?"을 예측

유방암 데이터셋: 종양의 측정값들로 **양성/악성(2개 범주)** 판정 → 타깃이 범주이므로 **분류**.

In [ ]:
cancer = load_breast_cancer()
Xc, yc = cancer.data, cancer.target

print('특성 개수:', Xc.shape[1])
print('타깃 클래스:', dict(zip(cancer.target_names, [0, 1])), '  <- 범주(카테고리)!')
print('클래스 분포:', np.bincount(yc), '(악성 개수, 양성 개수)')

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix

Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(Xc, yc, test_size=0.2, random_state=42, stratify=yc)

# 스케일링은 '훈련 데이터로만 fit' — 데이터 누수 방지 (4강 핵심!)
scaler = StandardScaler()
Xc_tr_s = scaler.fit_transform(Xc_tr)   # 훈련: fit + transform
Xc_te_s = scaler.transform(Xc_te)       # 테스트: transform만 (훈련 기준 적용)

clf = LogisticRegression(max_iter=5000)
clf.fit(Xc_tr_s, yc_tr)
pred_c = clf.predict(Xc_te_s)

print('정확도(accuracy):', round(accuracy_score(yc_te, pred_c), 3))

In [ ]:
# 분류 모델은 내부적으로 '확률'을 계산한 뒤 범주로 변환합니다 (2강 참고)
proba = clf.predict_proba(Xc_te_s)[:5]
for i, (p, label) in enumerate(zip(proba, pred_c[:5])):
    print(f'샘플{i}: 악성확률={p[0]:.2f}, 양성확률={p[1]:.2f} -> 최종판정={cancer.target_names[label]}')

# 혼동행렬 (로드맵 6번에서 자세히)
print('\n혼동행렬 [[TN, FP], [FN, TP]]:')
print(confusion_matrix(yc_te, pred_c))

## 정리

| | 회귀(Part A) | 분류(Part B) |
|---|---|---|
| 타깃 | 연속 숫자 | 범주 |
| 모델 | `LinearRegression` | `LogisticRegression` |
| 평가 | RMSE, R² | 정확도, 혼동행렬 |
| 공통 | **`fit` → `predict`** 흐름은 완전히 동일! | |

> 💡 scikit-learn의 모든 모델은 `fit`/`predict` API가 같습니다. 알고리즘만 바꿔 끼우면 됩니다.

## 🎯 직접 해보기

1. Part B에서 `LogisticRegression`을 `from sklearn.tree import DecisionTreeClassifier`의 `DecisionTreeClassifier()`로 바꿔보세요. 정확도가 어떻게 달라지나요? (API는 그대로!)
2. Part A에서 `test_size=0.2`를 `0.5`로 바꾸면 R²가 어떻게 변하나요? 왜 그럴까요?
3. `load_iris`(붓꽃 3종 분류)를 불러와 Part B를 따라 분류해 보세요. 이건 클래스가 3개인 **다중 분류**입니다.